# Data Cleaning
Prepare the raw German rental listings for EDA, SQL loading, and validation.


## Imports


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

C:\Users\mudry\AppData\Local\Temp\ipykernel_22372\2180542762.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Load Raw Data


In [2]:
# Load the raw rental listings from the project data folder.
BASE_DIR = Path.cwd().parent
csv_path = BASE_DIR / 'data' / 'raw_data.csv'

df = pd.read_csv(csv_path)

In [3]:
df.head()

,regio1,serviceCharge,heatingType,telekomTvOffer,telekomHybridUploadSpeed,newlyConst,balcony,picturecount,pricetrend,telekomUploadSpeed,...,regio2,regio3,description,facilities,heatingCosts,energyEfficiencyClass,lastRefurbish,electricityBasePrice,electricityKwhPrice,date
0,Nordrhein_Westfalen,245.00,central_heating,ONE_YEAR_FREE,NaN,False,False,6,4.62,10.0,...,Dortmund,Schüren,Die ebenerdig zu erreichende Erdgeschosswohnun...,Die Wohnung ist mit Laminat ausgelegt. Das Bad...,NaN,NaN,NaN,NaN,NaN,May19
1,Rheinland_Pfalz,134.00,self_contained_central_heating,ONE_YEAR_FREE,NaN,False,True,8,3.47,10.0,...,Rhein_Pfalz_Kreis,Böhl_Iggelheim,Alles neu macht der Mai – so kann es auch für ...,NaN,NaN,NaN,2019.0,NaN,NaN,May19
2,Sachsen,255.00,floor_heating,ONE_YEAR_FREE,10.0,True,True,8,2.72,2.4,...,Dresden,Äußere_Neustadt_Antonstadt,Der Neubau entsteht im Herzen der Dresdner Neu...,"* 9 m² Balkon\n* Bad mit bodengleicher Dusche,...",NaN,NaN,NaN,NaN,NaN,Oct19
3,Sachsen,58.15,district_heating,ONE_YEAR_FREE,NaN,False,True,9,1.53,40.0,...,Mittelsachsen_Kreis,Freiberg,Abseits von Lärm und Abgasen in Ihre neue Wohn...,NaN,87.23,NaN,NaN,NaN,NaN,May19
4,Bremen,138.00,self_contained_central_heating,NaN,NaN,False,True,19,2.46,NaN,...,Bremen,Neu_Schwachhausen,Es handelt sich hier um ein saniertes Mehrfami...,Diese Wohnung wurde neu saniert und ist wie fo...,NaN,NaN,NaN,NaN,NaN,Feb20


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268850 entries, 0 to 268849
Data columns (total 49 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   regio1                    268850 non-null  object 
 1   serviceCharge             261941 non-null  float64
 2   heatingType               223994 non-null  object 
 3   telekomTvOffer            236231 non-null  object 
 4   telekomHybridUploadSpeed  45020 non-null   float64
 5   newlyConst                268850 non-null  bool   
 6   balcony                   268850 non-null  bool   
 7   picturecount              268850 non-null  int64  
 8   pricetrend                267018 non-null  float64
 9   telekomUploadSpeed        235492 non-null  float64
 10  totalRent                 228333 non-null  float64
 11  yearConstructed           211805 non-null  float64
 12  scoutId                   268850 non-null  int64  
 13  noParkSpaces              93052 non-null   f

In [5]:
df.nunique()

regio1                          16
serviceCharge                12266
heatingType                     13
telekomTvOffer                   3
telekomHybridUploadSpeed         1
newlyConst                       2
balcony                          2
picturecount                    95
pricetrend                    1234
telekomUploadSpeed               7
totalRent                    28486
yearConstructed                465
scoutId                     268850
noParkSpaces                    71
firingTypes                    132
hasKitchen                       2
geo_bln                         16
cellar                           2
yearConstructedRange             9
baseRent                     26659
houseNumber                   5510
livingSpace                  13005
geo_krs                        419
condition                       10
interiorQual                     4
petsAllowed                      3
street                       52373
streetPlain                  54490
lift                

In [6]:
df['date'].astype(str).value_counts().head(20)

date
Feb20    79276
May19    76047
Oct19    66685
Sep18    46842
Name: count, dtype: int64

## Date Features


In [ ]:
# Parse the listing month field; invalid values become missing dates.
df['date'] = pd.to_datetime(df['date'], format='%b%y', errors='coerce')

In [ ]:
# Extract listing year and month 
df['listing_year'] = df['date'].dt.year
df['listing_month_name'] = df['date'].dt.month_name()

## Remove Unneeded Columns


In [9]:
# Drop identifiers, free-text fields, ranges, and provider-specific columns that are not needed for analysis.
drop_cols = [
'scoutId',
'houseNumber',
'street',
'streetPlain',
'description',
'facilities',
'electricityBasePrice',
'electricityKwhPrice',
'telekomHybridUploadSpeed',
'baseRentRange',
'noRoomsRange',
'livingSpaceRange',
'yearConstructedRange',
'picturecount',
'geo_bln',
'geo_krs',
'telekomUploadSpeed',
'telekomTvOffer',
'firingTypes',
'pricetrend',
'thermalChar'
]

df = df.drop(columns=drop_cols)

In [ ]:
# Summarize missing values to decide which sparse columns should be removed or inspected.
def missing_values(df,norows):   # input by the df and the number of rows that you want to show.
    total = df.isnull().sum().sort_values(ascending=False)
    percent = ((df.isnull().sum().sort_values(ascending=False)/df.shape[0])*100).sort_values(ascending=False)
    missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
    return(missing_data.head(norows))

missing_values(df,20)

,Total,Percent
energyEfficiencyClass,191063,71.066766
lastRefurbish,188139,69.979171
heatingCosts,183332,68.191185
noParkSpaces,175798,65.388879
petsAllowed,114573,42.615957
interiorQual,112665,41.906267
numberOfFloors,97732,36.351869
condition,68489,25.474800
yearConstructed,57045,21.218151
floor,51309,19.084620


In [ ]:
# Remove columns with too many missing values.
drop_cols = [
'energyEfficiencyClass',
'lastRefurbish',
'heatingCosts',
'noParkSpaces'
]

df = df.drop(columns=drop_cols)

## Required Values and Text Normalization


In [ ]:
df.dropna(subset=['totalRent'],inplace=True) # remove rows without rent.

In [ ]:
df = df[df['livingSpace'] > 0] # remove impossible area.

In [ ]:
# Standardize state names.
df.regio1 = df.regio1.str.replace('_', ' ').str.strip()

state_map = {
    'Nordrhein Westfalen': 'Nordrhein-Westfalen',
    'Baden Württemberg': 'Baden-Württemberg'
}

df['regio1'] = df['regio1'].replace(state_map)

In [ ]:
df.regio2 = df.regio2.str.replace('_', ' ').str.strip()

In [ ]:
df.regio3 = df.regio3.str.replace('_', ' ').str.strip()

In [ ]:
# Normalize heating type labels.
df['heatingType'] = (df['heatingType'].str.replace('_', ' ').str.title())

In [ ]:
# Normalize flat type labels.
df['typeOfFlat'] = (df['typeOfFlat'].str.lower().str.replace('_', ' ').str.title())

In [ ]:
# Normalize condition labels.
df['condition'] = (df['condition'].str.lower().str.replace('_', ' ').str.title())

In [ ]:
# Normalize interior quality labels.
df['interiorQual'] = (df['interiorQual'].str.title())

In [ ]:
# Normalize pet policy labels.
df['petsAllowed'] = (df['petsAllowed'].str.title())

## Outlier and Sanity Filters

In [ ]:
# Apply rent sanity filters and require warm rent to be above cold rent.
df = df[(df['baseRent'] > 200) & (df['baseRent'] < 8000)]
df = df[(df['totalRent'] > 200) & (df['totalRent'] < 9000)]
df = df[(df['totalRent'] > df['baseRent'])]
df = df[(df['totalRent'] - df['baseRent']) < 500]

In [ ]:
# Keep apartments within a plausible living-space range.
df = df[(df['livingSpace'] >= 15) & (df['livingSpace'] < 400)]

In [ ]:
# Keep listings with a plausible number of rooms.
df = df[(df['noRooms'] >= 1) & (df['noRooms'] <= 10)]

In [ ]:
# Treat impossible construction years as missing instead of deleting the full listing.
df.loc[df['yearConstructed'] < 1800, 'yearConstructed'] = np.nan
df.loc[df['yearConstructed'] > 2025, 'yearConstructed'] = np.nan

In [ ]:
# Treat implausible floor counts as missing values while preserving the listing.
df.loc[df['floor'] > 30, 'floor'] = np.nan
df.loc[df['numberOfFloors'] > 30, 'numberOfFloors'] = np.nan
df.loc[df['numberOfFloors'] == 0, 'numberOfFloors'] = np.nan

In [ ]:
# Keep service charges within a plausible non-negative range.
df = df[(df['serviceCharge'] >= 0) & (df['serviceCharge'] < 1000)]

## Derived Features


In [ ]:
# Create derived features used in EDA, SQL views, and validation.
df['price_per_m2'] = df['totalRent'] / df['livingSpace']
df['cold_rent_per_m2'] = df['baseRent'] / df['livingSpace']
df['property_age'] = df['listing_year'] - df['yearConstructed']
df.loc[df['property_age'] < 0, 'property_age'] = np.nan
df['room_density'] = df['livingSpace'] / df['noRooms']

In [ ]:
# Remove extreme price-per-square-meter values.
df = df[(df['price_per_m2'] > 2) & (df['price_per_m2'] < 60)]

In [ ]:
# Remove listings with implausibly high living-space-per-room ratios.
df = df[df['room_density'] < 80]

## Duplicate Check


In [ ]:
# Check duplicate listings using stable property and location attributes.
df.duplicated(subset=[
'livingSpace',
'noRooms',
'totalRent',
'regio3'
]).sum()

13385

In [ ]:
# Drop duplicate listings based on the same stable property and location attributes.
df = df.drop_duplicates(subset=[
'livingSpace',
'noRooms',
'totalRent',
'regio3'
])

## Final Checks and Export


In [ ]:
# Print the final cleaned dataset size before export.
print('Rows after cleaning:', len(df))
print('Columns:', len(df.columns))

Rows after cleaning: 197577
Columns: 30


In [ ]:
# Re-check missing values after cleaning to understand remaining data gaps.
missing_values(df, 20)

,Total,Percent
petsAllowed,80681,40.835219
interiorQual,74005,37.456283
numberOfFloors,66046,33.427980
condition,46372,23.470343
property_age,42307,21.412917
yearConstructed,41976,21.245388
floor,33104,16.754987
heatingType,28088,14.216230
typeOfFlat,23963,12.128436
date,0,0.000000


In [ ]:
# Inspect high-rent listings that remain after the outlier filters.
df[df['totalRent'] > 6000][['regio2','livingSpace','totalRent']].head(20)

,regio2,livingSpace,totalRent
969,München,143.00,7150.0
9482,Frankfurt am Main,170.00,6035.0
20548,München,185.00,6385.0
29878,München,300.00,7600.0
43916,München,242.00,7680.0
104969,München,198.00,6430.0
131735,Frankfurt am Main,345.00,6900.0
150596,München,180.52,6408.0
183777,Frankfurt am Main,168.00,6035.0
187090,München,288.00,6485.0


In [ ]:
df.columns = (df.columns.str.replace(r'([a-z])([A-Z])', r'\1_\2', regex=True).str.lower()) # camelCase → snake_case

In [ ]:
# Rename source geography and rent columns.
df = df.rename(columns={
    'regio1': 'state',
    'regio2': 'city',
    'regio3': 'district',
    'geo_plz': 'postal_code',
    'totalRent': 'warm_rent',
    'baseRent': 'cold_rent'
})

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 197577 entries, 0 to 268848
Data columns (total 30 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   state               197577 non-null  object        
 1   service_charge      197577 non-null  float64       
 2   heating_type        169489 non-null  object        
 3   newly_const         197577 non-null  bool          
 4   balcony             197577 non-null  bool          
 5   total_rent          197577 non-null  float64       
 6   year_constructed    155601 non-null  float64       
 7   has_kitchen         197577 non-null  bool          
 8   cellar              197577 non-null  bool          
 9   base_rent           197577 non-null  float64       
 10  living_space        197577 non-null  float64       
 11  condition           151205 non-null  object        
 12  interior_qual       123572 non-null  object        
 13  pets_allowed        116896 non-nul

In [ ]:
# Export the cleaned dataset.
df.to_csv('D:/work_space/germany_rent_analysis/data/rent_cleaned.csv', index=False)